# 04 — Feature Engineering

Turns the cleaned raw columns into features that map to something a hospital analyst could explain in one sentence. Implementation lives in `src/features.py` (reusable, tested) — this notebook runs it and checks the results make sense. See `src/features.py` docstrings for the full rationale per feature.

In [1]:
import sys
sys.path.insert(0, '../src')
import pandas as pd
from features import engineer_features

df = pd.read_csv('../data/processed/diabetic_data_clean.csv')
out = engineer_features(df)
print(f'{df.shape} -> {out.shape}')
new_cols = sorted(set(out.columns) - set(df.columns))
new_cols

(100114, 48) -> (100114, 62)


['age_midpoint',
 'comorbidity_burden',
 'diag_1_group',
 'diag_2_group',
 'diag_3_group',
 'has_circulatory_comorbidity',
 'has_diabetes_comorbidity',
 'high_utilizer',
 'hospitalization_intensity',
 'med_count_changed',
 'medication_complexity',
 'prior_emergency_flag',
 'prior_inpatient_flag',
 'prior_utilization']

## Diagnosis grouping

700+ raw ICD-9 codes per diagnosis field is too high-cardinality to model directly. Grouped into the standard 9 clinical categories used since Strack et al. (2014) on this exact dataset.

In [2]:
out['diag_1_group'].value_counts()

diag_1_group
Circulatory        29881
Other              17887
Respiratory        14074
Digestive           9380
Diabetes            8693
Injury              6881
Genitourinary       5054
Musculoskeletal     4944
Neoplasms           3299
Unknown               21
Name: count, dtype: int64

## Prior utilization score — do the new features actually separate the target?

In [3]:
for col in ['prior_utilization', 'high_utilizer', 'medication_complexity', 'hospitalization_intensity', 'comorbidity_burden']:
    print(f'--- {col} ---')
    print(out.groupby(out['readmitted_30d'])[col].mean().round(2))
    print()

--- prior_utilization ---
readmitted_30d
0    1.10
1    2.02
Name: prior_utilization, dtype: float64

--- high_utilizer ---
readmitted_30d
0    0.14
1    0.27
Name: high_utilizer, dtype: float64

--- medication_complexity ---
readmitted_30d
0    16.15
1    17.24
Name: medication_complexity, dtype: float64

--- hospitalization_intensity ---
readmitted_30d
0    211.05
1    236.80
Name: hospitalization_intensity, dtype: float64

--- comorbidity_burden ---
readmitted_30d
0    7.37
1    7.69
Name: comorbidity_burden, dtype: float64



Every engineered feature shows a higher mean among readmitted patients than non-readmitted, in the expected direction — a basic sanity check before these go into modeling.

## Rare categorical levels — deferred to the modeling pipeline, not done here

It's tempting to collapse rare categories (e.g. `medical_specialty` has specialties with under 20 patients) at this stage. We deliberately don't — doing it here, before the train/test split, would mean the "rare" designation is computed using test-set rows too (mild leakage). Instead `src/models.py` includes a `RareCategoryCollapser` that is fit only on the training fold, inside the pipeline. See `07_shap_analysis.ipynb` for why this mattered in practice: without it, SHAP importance was dominated by noise from categories with under 50 observations (e.g. a handful of `discharge_disposition_id` and medication Up/Down levels).

In [4]:
out.to_csv('../data/processed/diabetic_data_features.csv', index=False)
print('Saved data/processed/diabetic_data_features.csv', out.shape)

Saved data/processed/diabetic_data_features.csv (100114, 62)
